# Prueba de voz clonada — SlotBrain (Botbarber2)

Pipeline completo en Colab, costo $0 (GPU T4 gratis + Whisper local + Qwen3-TTS open source, Apache 2.0):

1. Cargar el sample de voz del dueño (15-30 s) + su transcripción exacta.
2. Sanidad: transcribir el sample con Whisper local y comparar contra la transcripción manual (si difiere, la clonación sale peor: corregir y repetir).
3. Clonar la voz con Qwen3-TTS-12Hz-0.6B-Base y generar frases reales del bot en español.
4. Convertir a ogg/opus mono (formato de WhatsApp) y verificar peso (< 512 KB = botón de play directo en el chat).
5. Círculo completo: subir una nota de voz de WhatsApp (ogg) → transcribirla → responderla con la voz clonada.

Antes de correr: Entorno de ejecución → Cambiar tipo de entorno → GPU T4.

Nota: Colab es solo para el experimento (la sesión se muere y los términos no permiten producción). Para producción seguimos con GPU serverless según el plan de voz.

In [ ]:
# @title 1. Instalación (unos 2-3 min)
!nvidia-smi -L
%pip install -q -U qwen-tts faster-whisper soundfile
import os
assert os.system("ffmpeg -version > /dev/null 2>&1") == 0, "ffmpeg no está (Colab normal lo trae)"
print("Listo. ffmpeg y librerías instaladas.")

In [ ]:
# @title 2. Cargar el sample de voz del dueño
# Sube el audio de referencia (wav, mp3, ogg o m4a; 15-30 s, habitación silenciosa).
# Si prefieres Google Drive: guarda REF_AUDIO = "/content/drive/MyDrive/ruta/sample.wav"
from google.colab import files

subido = files.upload()
REF_AUDIO = list(subido.keys())[0]

# Transcripción EXACTA palabra por palabra de lo que dice el sample.
# Edítala aquí (o reescríbela al correr la celda). Los errores aquí son lo que más degrada la clonación.
REF_TEXT = ""  # @param {type:"string"}
if not REF_TEXT:
    REF_TEXT = input("Escribe la transcripción exacta del sample: ").strip()

print(f"\nSample: {REF_AUDIO}")
print(f"Transcripción: {REF_TEXT}")

In [ ]:
# @title 3. Sanidad: Whisper local transcribe el sample y lo comparamos
# Si la transcripción de Whisper no coincide con REF_TEXT, corrige REF_TEXT en la celda 2 y vuelve a correr.
from faster_whisper import WhisperModel

dispositivo = "cuda"
try:
    wmodel = WhisperModel("small", device=dispositivo, compute_type="float16")
except Exception:
    dispositivo = "cpu"
    wmodel = WhisperModel("small", device="cpu", compute_type="int8")

segmentos, info = wmodel.transcribe(REF_AUDIO, language="es")
texto_wisper = " ".join(s.text.strip() for s in segmentos).strip()

print(f"Dispositivo: {dispositivo} | Duración: {info.duration:.1f}s")
print(f"Whisper dice : {texto_wisper}")
print(f"Tu texto dice: {REF_TEXT}")

In [ ]:
# @title 4. Cargar Qwen3-TTS y construir el prompt de voz clonada
# 0.6B-Base es el modelo de clonación (~2,5 GB, sobra en la T4).
# T4 no soporta bfloat16 y float16 rompe la clonación (NaN -> "probability tensor contains inf, nan"). Confirmado por el equipo de Qwen (issue #43): usar float32 en GPUs sin bfloat16. (sdpa en vez de flash-attn para no compilar 20 min en Colab.)
import torch
from qwen_tts import Qwen3TTSModel

dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float32
model = Qwen3TTSModel.from_pretrained(
    "Qwen/Qwen3-TTS-12Hz-0.6B-Base",
    device_map="cuda:0",
    dtype=dtype,
)

# Prompt reutilizable: se calcula UNA vez con el sample y se reusa en todas las frases.
voice_clone_prompt = model.create_voice_clone_prompt(
    ref_audio=REF_AUDIO,
    ref_text=REF_TEXT,
)
print(f"Modelo cargado (dtype={dtype}) y voz clonada lista.")

In [ ]:
# @title 5. Generar frases de prueba con tu voz (frases reales del bot)
import soundfile as sf
from IPython.display import Audio, display
from pathlib import Path

frases = [
    "Listo, te agendé el corte a las tres de la tarde.",
    "Señor, le recuerdo su cita de mañana a las diez con Andrés.",
    "Tenemos un hueco hoy a las cinco, ¿se lo aparto?",
    "Listo, quedé atento. ¡Hasta pronto!",
]

wavs, sr = model.generate_voice_clone(
    text=frases,
    language=["Spanish"] * len(frases),  # o "Auto" para que el modelo decida
    voice_clone_prompt=voice_clone_prompt,
)

Path("salidas").mkdir(exist_ok=True)
for i, (f, w) in enumerate(zip(frases, wavs)):
    ruta = f"salidas/frase_{i}.wav"
    sf.write(ruta, w, sr)
    print(f"\n[{i}] {f}")
    display(Audio(ruta))

In [ ]:
# @title 6. Convertir a ogg/opus mono (formato WhatsApp) y verificar peso
# WhatsApp exige ogg/opus, mono. Si pesa menos de 512 KB, el chat muestra botón de play directo.
import os
from IPython.display import Audio, display

for i in range(len(wavs)):
    ogg = f"salidas/frase_{i}.ogg"
    os.system(f'ffmpeg -y -loglevel error -i salidas/frase_{i}.wav -ac 1 -ar 48000 -c:a libopus -b:a 32k "{ogg}"')
    peso = os.path.getsize(ogg)
    marca = "OK (play directo en WhatsApp)" if peso < 512 * 1024 else "PESA MÁS DE 512 KB (llega como archivo)"
    print(f"frase_{i}.ogg: {peso/1024:.0f} KB — {marca}")
    display(Audio(ogg))

## 7. Círculo completo: nota de voz entrante → transcripción → respuesta con tu voz

Simula lo que hará el bot en producción: un cliente manda una nota de voz por WhatsApp (ogg), se transcribe y el bot responde con la voz clonada.

En esta primera vuelta la respuesta es fija (para aislar la prueba). El cerebro LLM se enchufa en la segunda vuelta.

In [ ]:
# @title 7a. Subir una nota de voz de WhatsApp (ogg) y transcribirla con Whisper local
from google.colab import files
from IPython.display import Audio, display

subido_ogg = files.upload()
NOTA_ENTRANTE = list(subido_ogg.keys())[0]

segmentos, info = wmodel.transcribe(NOTA_ENTRANTE, language="es")
texto_cliente = " ".join(s.text.strip() for s in segmentos).strip()

print(f"\nEl cliente dijo ({info.duration:.1f}s): {texto_cliente}")
display(Audio(NOTA_ENTRANTE))

In [ ]:
# @title 7b. Responder con la voz clonada y dejarla lista para WhatsApp
# Segunda vuelta: aquí entra el cerebro (Groq gpt-oss-120b, la misma key del bot) en vez de RESPUESTA_FIJA.
RESPUESTA_FIJA = "Listo, te agendé el corte a las tres de la tarde con Andrés. Quedé atento."

wavs_resp, sr_resp = model.generate_voice_clone(
    text=RESPUESTA_FIJA,
    language="Spanish",
    voice_clone_prompt=voice_clone_prompt,
)
sf.write("salidas/respuesta.wav", wavs_resp[0], sr_resp)
os.system('ffmpeg -y -loglevel error -i salidas/respuesta.wav -ac 1 -ar 48000 -c:a libopus -b:a 32k "salidas/respuesta.ogg"')

peso = os.path.getsize("salidas/respuesta.ogg")
print(f"respuesta.ogg: {peso/1024:.0f} KB — {'OK para WhatsApp' if peso < 512*1024 else 'revisar peso'}")
display(Audio("salidas/respuesta.ogg"))

## 8. Opcional: transcripción con Groq (modo rápido)

Whisper local ya es gratis y corre en la GPU de este notebook. Groq (whisper-large-v3-turbo, free tier, la misma key del bot) es más rápido y útil si el audio es largo. Dejar vacío = se usa el local.

In [ ]:
# @title 8a. (Opcional) Transcribir con Groq whisper-large-v3-turbo
import os, getpass

GROQ_API_KEY = getpass.getpass("GROQ_API_KEY (enter para omitir y usar Whisper local): ").strip()

if GROQ_API_KEY:
    import requests
    with open(NOTA_ENTRANTE, "rb") as f:
        r = requests.post(
            "https://api.groq.com/openai/v1/audio/transcriptions",
            headers={"Authorization": f"Bearer {GROQ_API_KEY}"},
            files={"file": (NOTA_ENTRANTE, f, "audio/ogg")},
            data={"model": "whisper-large-v3-turbo", "language": "es"},
            timeout=120,
        )
    r.raise_for_status()
    print("Groq dice:", r.json()["text"])
else:
    print("Sin key: se usa Whisper local (gratis).")

In [ ]:
# @title 9. (Opcional) Descargar los audios generados
# files.download("salidas/frase_0.ogg")
files.download("salidas/respuesta.ogg")